# POCR-EXP-005 — YOLO11n Plate Detector Fine-Tune

Bu notebook Anomali Road Safety AI projesi için kapsamlı **plaka tespiti** modelini hazırlar.

Amaç smoke test değildir. Hedef:

1. Roboflow `Turkish Number Plates` ve `License Plate Recognition` veri setlerini indir.
2. Verileri tek sınıf `license_plate` formatına normalize et.
3. Duplicate / near-duplicate kontrolü yap.
4. Deterministik train/val/test split üret.
5. YOLO11n single-class plate detector fine-tune et.
6. Mevcut repo baseline'ı olan `license_plate_detector.pt` ile karşılaştır.
7. UFPR-ALPR mevcutsa external benchmark olarak değerlendir.
8. Best `.pt`, opsiyonel ONNX, metrik JSON/CSV ve raporları Drive'a kaydet.

Bu notebook OCR yapmaz. OCR'a geçiş için önce plate bbox ve usable crop kalitesi ölçülmelidir.

## Senin Yapman Gerekenler

1. Colab Runtime: **A100** seç. A100 yoksa L4 de olur, ama batch/epoch süresi daha uzun olur.
2. Google Drive mount iznini ver.
3. Colab Secrets içine şu secret'ı ekle:
   - `ROBOFLOW_API_KEY`
4. Roboflow veri setleri otomatik indirilecek:
   - Turkish Number Plates: `plakatanima-vnt3k/turkish-number-plates`, version `2`
   - License Plate Recognition: `roboflow-universe-projects/license-plate-recognition-rxg4e`, version `13`
5. UFPR-ALPR otomatik indirilmeyecek. Resmi lisans akademik/non-commercial kullanım ve erişim onayı ister. İzinli zip elde edilirse Drive'da şu yollardan birine koy:
   - `/content/drive/MyDrive/anomali-road-safety-ai/datasets/plate_detection/raw/manual_zips/ufpr_alpr.zip`
   - veya extracted klasör: `/content/drive/MyDrive/anomali-road-safety-ai/datasets/plate_detection/raw/ufpr_alpr/`
6. Notebook sonunda Drive'da şu çıktıları bekle:
   - `runs/plate_detection/POCR-EXP-005/`
   - `models/checkpoints/plate/POCR-EXP-005-YOLO11N-PLATE-DETECTOR-best.pt`
   - `models/benchmarks/artifacts/plate_detection/POCR-EXP-005-summary.json`
   - `models/experiments/POCR_EXP_005_plate_detector_report.md`

API key veya model dosyalarını Git'e ekleme.

In [ ]:
# Cell 1 - Dependencies
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics>=8.3.0', 'roboflow', 'pyyaml', 'pandas', 'scikit-learn',
    'opencv-python-headless', 'pillow', 'tqdm', 'imagehash', 'huggingface_hub', 'tabulate',
])


0

In [ ]:
# Cell 2 - Imports, Drive mount, config
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import yaml
from PIL import Image
from tqdm.auto import tqdm

try:
    from google.colab import drive, userdata
except Exception:
    drive = None
    userdata = None

if drive is not None:
    drive.mount('/content/drive')

EXPERIMENT_ID = 'POCR-EXP-005-YOLO11N-PLATE-DETECTOR'
SEED = 42
random.seed(SEED)

PROJECT_ROOT = Path('/content/drive/MyDrive/anomali-road-safety-ai')
LOCAL_ROOT = Path('/content/anomali-road-safety-ai-work')

DRIVE_RAW_ROOT = PROJECT_ROOT / 'datasets' / 'plate_detection' / 'raw'
DRIVE_DOWNLOAD_MANIFEST_ROOT = DRIVE_RAW_ROOT / 'download_manifests'
DRIVE_YOLO_ROOT = PROJECT_ROOT / 'datasets' / 'plate_detection_yolo'
DRIVE_RUN_ROOT = PROJECT_ROOT / 'runs' / 'plate_detection' / 'POCR-EXP-005'
DRIVE_ARTIFACT_ROOT = PROJECT_ROOT / 'models' / 'benchmarks' / 'artifacts' / 'plate_detection'
DRIVE_CHECKPOINT_ROOT = PROJECT_ROOT / 'models' / 'checkpoints' / 'plate'
DRIVE_EXPERIMENT_ROOT = PROJECT_ROOT / 'models' / 'experiments'

LOCAL_RAW_ROOT = LOCAL_ROOT / 'datasets' / 'plate_detection' / 'raw'
LOCAL_YOLO_ROOT = LOCAL_ROOT / 'datasets' / 'plate_detection_yolo'
LOCAL_RUN_ROOT = LOCAL_ROOT / 'runs' / 'plate_detection' / 'POCR-EXP-005'

for p in [PROJECT_ROOT, LOCAL_ROOT, DRIVE_RAW_ROOT, DRIVE_DOWNLOAD_MANIFEST_ROOT, DRIVE_YOLO_ROOT, DRIVE_RUN_ROOT,
          DRIVE_ARTIFACT_ROOT, DRIVE_CHECKPOINT_ROOT, DRIVE_EXPERIMENT_ROOT,
          LOCAL_RAW_ROOT, LOCAL_YOLO_ROOT, LOCAL_RUN_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# Training config. A100 için bu ayarlar makul başlangıçtır.
IMGSZ = 640
EPOCHS = 80
PATIENCE = 20
BATCH = 48          # A100: 48/64 denenebilir. L4/T4 için 16/24 yap.
WORKERS = 4
DEVICE = 0
CACHE_IMAGES = False  # RAM yeterliyse True denenebilir; Drive değil local path kullanıldığı için False güvenli.
RUN_ONNX_EXPORT = True

# Drive raw-tree caching is intentionally disabled. Roboflow LPR extracts 200k+ small files;
# copying that tree into Google Drive mount is slow and can raise Errno 5 I/O errors.
USE_DRIVE_RAW_TREE_CACHE = False
SAVE_RAW_TREE_TO_DRIVE = False

# Dataset config
ROBOFLOW_FORMAT = 'yolov8'
ROB0FLOW_DATASETS = [
    {
        'source_id': 'turkish_number_plates',
        'workspace': 'plakatanima-vnt3k',
        'project': 'turkish-number-plates',
        'version': 2,
        'format': ROBOFLOW_FORMAT,
        'usage': 'primary_train',
        'expected_classes': ['license_plate'],
        'url': 'https://universe.roboflow.com/plakatanima-vnt3k/turkish-number-plates',
    },
    {
        'source_id': 'roboflow_lpr',
        'workspace': 'roboflow-universe-projects',
        'project': 'license-plate-recognition-rxg4e',
        'version': 13,
        'format': ROBOFLOW_FORMAT,
        'usage': 'support_train',
        'expected_classes': ['License_Plate', 'license_plate'],
        'url': 'https://universe.roboflow.com/roboflow-universe-projects/license-plate-recognition-rxg4e',
    },
]

# Split policy after merge and dedup
TRAIN_PCT = 0.80
VAL_PCT = 0.10
TEST_PCT = 0.10

BASELINE_PLATE_CHECKPOINT = DRIVE_CHECKPOINT_ROOT / 'license_plate_detector.pt'
BASELINE_HF_REPO = 'morsetechlab/yolov11-license-plate-detection'
BASELINE_HF_FILENAME = 'license-plate-finetune-v1n.pt'
DOWNLOAD_BASELINE_IF_MISSING = True

BEST_CHECKPOINT_DRIVE = DRIVE_CHECKPOINT_ROOT / f'{EXPERIMENT_ID}-best.pt'
LAST_CHECKPOINT_DRIVE = DRIVE_CHECKPOINT_ROOT / f'{EXPERIMENT_ID}-last.pt'
SUMMARY_JSON_DRIVE = DRIVE_ARTIFACT_ROOT / f'{EXPERIMENT_ID}-summary.json'
METADATA_CSV_DRIVE = DRIVE_ARTIFACT_ROOT / f'{EXPERIMENT_ID}-dataset-metadata.csv'
REPORT_MD_DRIVE = DRIVE_EXPERIMENT_ROOT / 'POCR_EXP_005_plate_detector_report.md'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('LOCAL_ROOT:', LOCAL_ROOT)
print('DRIVE_RAW_ROOT:', DRIVE_RAW_ROOT)
print('LOCAL_YOLO_ROOT:', LOCAL_YOLO_ROOT)
print('RUN_ROOT:', DRIVE_RUN_ROOT)
print('A100-oriented config:', {'imgsz': IMGSZ, 'epochs': EPOCHS, 'batch': BATCH, 'device': DEVICE})

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/anomali-road-safety-ai
LOCAL_ROOT: /content/anomali-road-safety-ai-work
DRIVE_RAW_ROOT: /content/drive/MyDrive/anomali-road-safety-ai/datasets/plate_detection/raw
LOCAL_YOLO_ROOT: /content/anomali-road-safety-ai-work/datasets/plate_detection_yolo
RUN_ROOT: /content/drive/MyDrive/anomali-road-safety-ai/runs/plate_detection/POCR-EXP-005
A100-oriented config: {'imgsz': 640, 'epochs': 80, 'batch': 48, 'device': 0}


In [ ]:
# Cell 3 - Preflight: GPU, Drive, API key, required folders

def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()


def get_secret(name: str) -> str | None:
    value = os.environ.get(name)
    if value:
        return value
    if userdata is not None:
        try:
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    return None

RF_API_KEY = get_secret('ROBOFLOW_API_KEY')

print('Timestamp UTC:', now_utc())
print('Drive project exists:', PROJECT_ROOT.exists())
print('Roboflow key present:', bool(RF_API_KEY))
print('\nGPU status:')
try:
    subprocess.run(['nvidia-smi'], check=False)
except Exception as exc:
    print('nvidia-smi unavailable:', exc)

if not RF_API_KEY:
    raise RuntimeError(
        'ROBOFLOW_API_KEY missing. Colab > Secrets içine ROBOFLOW_API_KEY ekle. '
        'Alternatif: Drive raw/manual_zips altına export edilmiş YOLOv8 zip dosyalarını koy.'
    )

for cfg in ROB0FLOW_DATASETS:
    print('dataset:', cfg['source_id'], cfg['url'], 'version', cfg['version'])

print('\nManual zip fallback folder:')
print(DRIVE_RAW_ROOT / 'manual_zips')
(DRIVE_RAW_ROOT / 'manual_zips').mkdir(parents=True, exist_ok=True)

Timestamp UTC: 2026-06-16T10:00:36.213339+00:00
Drive project exists: True
Roboflow key present: True

GPU status:
dataset: turkish_number_plates https://universe.roboflow.com/plakatanima-vnt3k/turkish-number-plates version 2
dataset: roboflow_lpr https://universe.roboflow.com/roboflow-universe-projects/license-plate-recognition-rxg4e version 13

Manual zip fallback folder:
/content/drive/MyDrive/anomali-road-safety-ai/datasets/plate_detection/raw/manual_zips


In [ ]:
# Cell 4 - Download Roboflow datasets with local-first, no raw-tree Drive copy
from roboflow import Roboflow

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def has_data_yaml(root: Path) -> bool:
    return any(root.rglob('data.yaml')) or any(root.rglob('data.yml'))


def find_data_yaml(root: Path) -> Path | None:
    candidates = sorted(list(root.rglob('data.yaml')) + list(root.rglob('data.yml')))
    return candidates[0] if candidates else None


def safe_copytree(src: Path, dst: Path) -> None:
    if not src.exists():
        return
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)


def unzip_to(zip_path: Path, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    marker = out_dir / '.extracted.marker'
    if marker.exists() and has_data_yaml(out_dir):
        print('[skip extract]', zip_path.name, 'already extracted ->', out_dir)
        return
    print('[extract]', zip_path, '->', out_dir)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out_dir)
    marker.write_text(now_utc(), encoding='utf-8')


def drive_cache_marker(drive_dir: Path) -> Path:
    return drive_dir / '.roboflow_tree_cache_complete.json'


def has_complete_drive_tree_cache(drive_dir: Path) -> bool:
    return USE_DRIVE_RAW_TREE_CACHE and drive_cache_marker(drive_dir).exists() and has_data_yaml(drive_dir)


def write_download_manifest(source_id: str, method: str, local_dir: Path, extra: dict | None = None) -> None:
    DRIVE_DOWNLOAD_MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
    manifest = {
        'source_id': source_id,
        'method': method,
        'timestamp_utc': now_utc(),
        'local_dir': str(local_dir),
        'data_yaml': str(find_data_yaml(local_dir)) if find_data_yaml(local_dir) else None,
        'raw_tree_copied_to_drive': False,
        'note': 'Raw Roboflow image/label tree is not copied to Drive because Drive mount can fail on hundreds of thousands of small files. Final checkpoints/metadata are persisted instead.',
    }
    if extra:
        manifest.update(extra)
    (DRIVE_DOWNLOAD_MANIFEST_ROOT / f'{source_id}.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')


def maybe_save_raw_tree_to_drive(source_id: str, local_dir: Path, drive_dir: Path) -> None:
    if not SAVE_RAW_TREE_TO_DRIVE:
        print('[skip raw tree Drive copy]', source_id, '- using local runtime dataset; final artifacts will persist to Drive')
        return
    print('[local -> drive raw tree cache]', source_id, '- this can be slow')
    try:
        safe_copytree(local_dir, drive_dir)
        drive_cache_marker(drive_dir).write_text(json.dumps({'source_id': source_id, 'timestamp_utc': now_utc()}, indent=2), encoding='utf-8')
    except Exception as exc:
        print('[warn] raw tree Drive copy failed; continuing with local dataset:', repr(exc))


def ensure_roboflow_dataset(cfg: dict) -> Path:
    source_id = cfg['source_id']
    local_dir = LOCAL_RAW_ROOT / source_id
    drive_dir = DRIVE_RAW_ROOT / source_id
    manual_zip = DRIVE_RAW_ROOT / 'manual_zips' / f'{source_id}_yolov8.zip'

    if has_data_yaml(local_dir):
        print('[local exists]', source_id, local_dir)
        write_download_manifest(source_id, 'local_exists', local_dir)
        return local_dir

    if has_complete_drive_tree_cache(drive_dir):
        print('[complete drive tree cache -> local]', source_id)
        safe_copytree(drive_dir, local_dir)
        write_download_manifest(source_id, 'complete_drive_tree_cache', local_dir)
        return local_dir

    if has_data_yaml(drive_dir) and not drive_cache_marker(drive_dir).exists():
        print('[ignore partial/incomplete Drive raw tree cache]', source_id, drive_dir)
        print('  Reason: no .roboflow_tree_cache_complete.json marker. This avoids stale/partial Drive copies after Errno 5 failures.')

    if manual_zip.exists():
        print('[manual zip -> local]', source_id, manual_zip)
        unzip_to(manual_zip, local_dir)
        if not has_data_yaml(local_dir):
            raise FileNotFoundError(f'Manual zip extracted but no data.yaml found under {local_dir}')
        write_download_manifest(source_id, 'manual_zip', local_dir, {'manual_zip': str(manual_zip)})
        maybe_save_raw_tree_to_drive(source_id, local_dir, drive_dir)
        return local_dir

    print('[roboflow download]', source_id, cfg['workspace'], cfg['project'], 'v', cfg['version'])
    rf = Roboflow(api_key=RF_API_KEY)
    project = rf.workspace(cfg['workspace']).project(cfg['project'])
    version = project.version(int(cfg['version']))
    try:
        dataset = version.download(cfg['format'], location=str(local_dir), overwrite=False)
    except TypeError:
        dataset = version.download(cfg['format'], location=str(local_dir))
    print('Roboflow dataset location:', getattr(dataset, 'location', local_dir))

    if not has_data_yaml(local_dir):
        raise FileNotFoundError(f'Download completed but no data.yaml found under {local_dir}')

    write_download_manifest(source_id, 'roboflow_api_download', local_dir, {'roboflow_url': cfg.get('url')})
    maybe_save_raw_tree_to_drive(source_id, local_dir, drive_dir)
    return local_dir

raw_dataset_dirs = {}
for cfg in ROB0FLOW_DATASETS:
    raw_dataset_dirs[cfg['source_id']] = ensure_roboflow_dataset(cfg)
    print(cfg['source_id'], 'data.yaml ->', find_data_yaml(raw_dataset_dirs[cfg['source_id']]))


[ignore partial/incomplete Drive raw tree cache] turkish_number_plates /content/drive/MyDrive/anomali-road-safety-ai/datasets/plate_detection/raw/turkish_number_plates
  Reason: no .roboflow_tree_cache_complete.json marker. This avoids stale/partial Drive copies after Errno 5 failures.
[roboflow download] turkish_number_plates plakatanima-vnt3k turkish-number-plates v 2
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/anomali-road-safety-ai-work/datasets/plate_detection/raw/turkish_number_plates in yolov8:: 100%|██████████| 10980/10980 [00:01<00:00, 8063.23it/s] 


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Roboflow dataset location: /content/anomali-road-safety-ai-work/datasets/plate_detection/raw/turkish_number_plates
[skip raw tree Drive copy] turkish_number_plates - using local runtime dataset; final artifacts will persist to Drive
turkish_number_plates data.yaml -> /content/anomali-road-safety-ai-work/datasets/plate_detection/raw/turkish_number_plates/data.yaml
[roboflow download] roboflow_lpr roboflow-universe-projects license-plate-recognition-rxg4e v 13
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/anomali-road-safety-ai-work/datasets/plate_detection/raw/roboflow_lpr in yolov8:: 100%|██████████| 203744/203744 [00:22<00:00, 9166.28it/s] 


Roboflow dataset location: /content/anomali-road-safety-ai-work/datasets/plate_detection/raw/roboflow_lpr
[skip raw tree Drive copy] roboflow_lpr - using local runtime dataset; final artifacts will persist to Drive
roboflow_lpr data.yaml -> /content/anomali-road-safety-ai-work/datasets/plate_detection/raw/roboflow_lpr/data.yaml


In [ ]:
# Cell 5 - Validate raw datasets and collect source records

def read_yaml(path: Path) -> dict:
    with path.open('r', encoding='utf-8') as f:
        return yaml.safe_load(f)


def normalize_names(names):
    if isinstance(names, dict):
        return {int(k): str(v) for k, v in names.items()}
    if isinstance(names, list):
        return {i: str(v) for i, v in enumerate(names)}
    return {}


def split_aliases(split_name: str) -> list[str]:
    if split_name == 'train':
        return ['train', 'training']
    if split_name == 'val':
        return ['val', 'valid', 'validation']
    if split_name == 'test':
        return ['test', 'testing']
    return [split_name]


def unique_existing_dirs(paths: list[Path]) -> list[Path]:
    out = []
    seen = set()
    for p in paths:
        try:
            rp = p.resolve()
        except Exception:
            rp = p
        if rp.exists() and rp.is_dir() and str(rp) not in seen:
            out.append(rp)
            seen.add(str(rp))
    return out


def candidate_dirs_from_yaml_value(root: Path, raw_value) -> list[Path]:
    if raw_value is None:
        return []
    values = raw_value if isinstance(raw_value, list) else [raw_value]
    cands = []
    for val in values:
        p = Path(str(val))
        if p.is_absolute():
            cands.append(p)
        else:
            # Roboflow exports may write ../train/images even when train/images is under the dataset root.
            cands.append(root / p)
            cleaned_parts = [part for part in p.parts if part not in ('.', '..')]
            if cleaned_parts:
                cands.append(root / Path(*cleaned_parts))
            cands.append(root.parent / p)
    return cands


def find_split_image_dirs(root: Path, data_cfg: dict, split_name: str) -> list[Path]:
    aliases = split_aliases(split_name)
    raw_values = []
    for key in [split_name, *aliases]:
        if key in data_cfg:
            raw_values.append(data_cfg.get(key))
    candidates = []
    for raw in raw_values:
        candidates.extend(candidate_dirs_from_yaml_value(root, raw))

    # Common Roboflow layouts.
    for alias in aliases:
        candidates.extend([
            root / alias / 'images',
            root / alias,
            root / f'{alias}/images',
        ])

    # Fallback discovery: any images dir under a split alias.
    for img_dir in root.rglob('images'):
        parts = {part.lower() for part in img_dir.parts}
        parent = img_dir.parent.name.lower()
        if parent in aliases or parts.intersection(set(aliases)):
            candidates.append(img_dir)

    dirs = unique_existing_dirs(candidates)
    # Keep only dirs that actually contain image files.
    dirs = [d for d in dirs if any(p.suffix.lower() in IMAGE_EXTS for p in d.rglob('*'))]
    return dirs


def image_to_label_path(img_path: Path) -> Path | None:
    parts = list(img_path.parts)
    candidates = []
    for i, part in enumerate(parts):
        if part == 'images':
            candidates.append(Path(*parts[:i], 'labels', *parts[i+1:]).with_suffix('.txt'))
    candidates.extend([
        img_path.parent.parent / 'labels' / f'{img_path.stem}.txt',
        img_path.parent / 'labels' / f'{img_path.stem}.txt',
        img_path.with_suffix('.txt'),
    ])
    for cand in candidates:
        if cand.exists() and cand.is_file():
            return cand
    return None


def tree_preview(root: Path, max_items: int = 80) -> None:
    print('Tree preview:', root)
    count = 0
    for p in sorted(root.rglob('*')):
        rel = p.relative_to(root)
        # Keep preview shallow and focused.
        if len(rel.parts) <= 3:
            print(' ', 'dir ' if p.is_dir() else 'file', rel)
            count += 1
        if count >= max_items:
            break

source_summaries = []
source_records = []

for cfg in ROB0FLOW_DATASETS:
    source_id = cfg['source_id']
    root = raw_dataset_dirs[source_id]
    yml = find_data_yaml(root)
    if yml is None:
        raise FileNotFoundError(f'{source_id}: data.yaml not found under {root}')
    data_cfg = read_yaml(yml)
    names = normalize_names(data_cfg.get('names', {}))
    print('\nDATASET', source_id)
    print(' root:', root)
    print(' data_yaml:', yml)
    print(' yaml split values:', {k: data_cfg.get(k) for k in ['train', 'val', 'valid', 'test'] if k in data_cfg})
    print(' names:', names)

    split_counts = {}
    split_label_counts = {}
    for split in ['train', 'val', 'test']:
        split_dirs = find_split_image_dirs(root, data_cfg, split)
        print(f' {split} image dirs:', [str(d) for d in split_dirs])
        images = []
        seen_images = set()
        for split_dir in split_dirs:
            for img in split_dir.rglob('*'):
                if img.suffix.lower() not in IMAGE_EXTS:
                    continue
                key = str(img.resolve())
                if key not in seen_images:
                    images.append(img)
                    seen_images.add(key)
        split_counts[split] = len(images)
        labeled = 0
        for img in images:
            label = image_to_label_path(img)
            if label is None:
                continue
            labeled += 1
            source_records.append({
                'source_id': source_id,
                'source_url': cfg['url'],
                'source_usage': cfg['usage'],
                'orig_split': split,
                'image_path': str(img),
                'label_path': str(label),
                'data_yaml': str(yml),
                'class_names': json.dumps(names, ensure_ascii=False),
            })
        split_label_counts[f'{split}_labeled'] = labeled
    source_summaries.append({'source_id': source_id, 'data_yaml': str(yml), 'names': names, **split_counts, **split_label_counts})
    print(' split_counts:', split_counts)
    print(' labeled_counts:', split_label_counts)
    if sum(split_counts.values()) == 0:
        tree_preview(root)

records_df = pd.DataFrame(source_records)
print('\nsource_records:', records_df.shape)
display(pd.DataFrame(source_summaries))
if records_df.empty:
    raise RuntimeError(
        'No labeled images found after robust Roboflow path discovery. '
        'The download succeeded, but labels/images were not paired. Check tree preview and manual zip export format.'
    )



DATASET turkish_number_plates
 root: /content/anomali-road-safety-ai-work/datasets/plate_detection/raw/turkish_number_plates
 data_yaml: /content/anomali-road-safety-ai-work/datasets/plate_detection/raw/turkish_number_plates/data.yaml
 yaml split values: {'train': '../train/images', 'val': '../valid/images', 'test': '../test/images'}
 names: {0: 'license_plate'}
 train image dirs: ['/content/anomali-road-safety-ai-work/datasets/plate_detection/raw/turkish_number_plates/train/images', '/content/anomali-road-safety-ai-work/datasets/plate_detection/raw/turkish_number_plates/train']
 val image dirs: ['/content/anomali-road-safety-ai-work/datasets/plate_detection/raw/turkish_number_plates/valid/images', '/content/anomali-road-safety-ai-work/datasets/plate_detection/raw/turkish_number_plates/valid']
 test image dirs: ['/content/anomali-road-safety-ai-work/datasets/plate_detection/raw/turkish_number_plates/test/images', '/content/anomali-road-safety-ai-work/datasets/plate_detection/raw/turki

,source_id,data_yaml,names,train,val,test,train_labeled,val_labeled,test_labeled
0,turkish_number_plates,/content/anomali-road-safety-ai-work/datasets/...,{0: 'license_plate'},4857,419,208,4857,419,208
1,roboflow_lpr,/content/anomali-road-safety-ai-work/datasets/...,{0: 'License_Plate'},98798,2048,1020,98798,2048,1020


In [ ]:
# Cell 6 - Normalize labels, exact/pHash dedup, deterministic split, YOLO dataset generation
import imagehash

LOCAL_ALL_IMAGES = LOCAL_YOLO_ROOT / 'images' / 'all'
LOCAL_ALL_LABELS = LOCAL_YOLO_ROOT / 'labels' / 'all'
for p in [LOCAL_ALL_IMAGES, LOCAL_ALL_LABELS]:
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)


def sha1_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha1()
    with path.open('rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def read_yolo_labels(label_path: Path) -> list[str]:
    rows = []
    for line in label_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 5:
            continue
        try:
            _, x, y, w, h = parts[:5]
            vals = [float(x), float(y), float(w), float(h)]
        except Exception:
            continue
        if not all(math.isfinite(v) for v in vals):
            continue
        # Clamp slightly out-of-range export noise.
        vals = [max(0.0, min(1.0, v)) for v in vals]
        if vals[2] <= 0 or vals[3] <= 0:
            continue
        rows.append('0 ' + ' '.join(f'{v:.6f}' for v in vals))
    return rows


def deterministic_split(key: str) -> str:
    # Stable source/hash split. 80/10/10.
    h = int(hashlib.sha1(key.encode('utf-8')).hexdigest(), 16) % 10000
    frac = h / 10000.0
    if frac < TRAIN_PCT:
        return 'train'
    if frac < TRAIN_PCT + VAL_PCT:
        return 'val'
    return 'test'

metadata_rows = []
seen_sha1 = set()
seen_phash = set()

for idx, row in tqdm(records_df.iterrows(), total=len(records_df), desc='normalize+dedup'):
    img_path = Path(row['image_path'])
    label_path = Path(row['label_path'])
    if not img_path.exists() or not label_path.exists():
        continue
    labels = read_yolo_labels(label_path)
    if not labels:
        continue
    try:
        img_sha1 = sha1_file(img_path)
    except Exception:
        continue
    if img_sha1 in seen_sha1:
        continue

    try:
        with Image.open(img_path) as im:
            width, height = im.size
            phash = str(imagehash.phash(im.convert('RGB')))
    except Exception:
        width = height = None
        phash = None

    # Exact pHash duplicate is intentionally conservative; no O(n^2) hamming matching.
    if phash and phash in seen_phash:
        continue

    seen_sha1.add(img_sha1)
    if phash:
        seen_phash.add(phash)

    source_id = row['source_id']
    unique_stem = f"{source_id}_{img_sha1[:16]}"
    out_img = LOCAL_ALL_IMAGES / f"{unique_stem}{img_path.suffix.lower()}"
    out_label = LOCAL_ALL_LABELS / f"{unique_stem}.txt"
    shutil.copy2(img_path, out_img)
    out_label.write_text('\n'.join(labels) + '\n', encoding='utf-8')
    split = deterministic_split(f"{source_id}:{img_sha1}")
    metadata_rows.append({
        'image_name': out_img.name,
        'label_name': out_label.name,
        'source_id': source_id,
        'source_usage': row['source_usage'],
        'orig_split': row['orig_split'],
        'split': split,
        'image_sha1': img_sha1,
        'image_phash': phash,
        'width': width,
        'height': height,
        'bbox_count': len(labels),
        'local_image_path': str(out_img),
        'local_label_path': str(out_label),
    })

metadata_df = pd.DataFrame(metadata_rows)
print('metadata rows:', metadata_df.shape)
if metadata_df.empty:
    raise RuntimeError('Merged YOLO dataset produced zero rows.')

# Materialize split folders.
for split in ['train', 'val', 'test']:
    for kind in ['images', 'labels']:
        d = LOCAL_YOLO_ROOT / kind / split
        if d.exists():
            shutil.rmtree(d)
        d.mkdir(parents=True, exist_ok=True)

for row in tqdm(metadata_rows, desc='copy split files'):
    split = row['split']
    src_img = Path(row['local_image_path'])
    src_lbl = Path(row['local_label_path'])
    shutil.copy2(src_img, LOCAL_YOLO_ROOT / 'images' / split / src_img.name)
    shutil.copy2(src_lbl, LOCAL_YOLO_ROOT / 'labels' / split / src_lbl.name)

# Persist metadata local and Drive.
LOCAL_YOLO_ROOT.mkdir(parents=True, exist_ok=True)
metadata_csv_local = LOCAL_YOLO_ROOT / 'pocr_exp_005_plate_metadata.csv'
metadata_df.to_csv(metadata_csv_local, index=False)
METADATA_CSV_DRIVE.parent.mkdir(parents=True, exist_ok=True)
metadata_df.to_csv(METADATA_CSV_DRIVE, index=False)

print('split distribution:')
display(metadata_df.groupby(['split', 'source_id']).agg(images=('image_name', 'count'), boxes=('bbox_count', 'sum')).reset_index())
print('metadata local:', metadata_csv_local)
print('metadata drive:', METADATA_CSV_DRIVE)

normalize+dedup:   0%|          | 0/107350 [00:00<?, ?it/s]

metadata rows: (106432, 13)


copy split files:   0%|          | 0/106432 [00:00<?, ?it/s]

split distribution:


,split,source_id,images,boxes
0,test,roboflow_lpr,10159,10591
1,test,turkish_number_plates,598,629
2,train,roboflow_lpr,80707,84199
3,train,turkish_number_plates,4332,4566
4,val,roboflow_lpr,10113,10533
5,val,turkish_number_plates,523,553


metadata local: /content/anomali-road-safety-ai-work/datasets/plate_detection_yolo/pocr_exp_005_plate_metadata.csv
metadata drive: /content/drive/MyDrive/anomali-road-safety-ai/models/benchmarks/artifacts/plate_detection/POCR-EXP-005-YOLO11N-PLATE-DETECTOR-dataset-metadata.csv


In [ ]:
# Cell 7 - Write YOLO data.yaml and run pre-training sanity checks
DATA_YAML_LOCAL = LOCAL_YOLO_ROOT / 'data.yaml'
DATA_YAML_DRIVE = DRIVE_YOLO_ROOT / 'data.yaml'

data_yaml = {
    'path': str(LOCAL_YOLO_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 1,
    'names': {0: 'license_plate'},
}
DATA_YAML_LOCAL.write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding='utf-8')
DRIVE_YOLO_ROOT.mkdir(parents=True, exist_ok=True)
DATA_YAML_DRIVE.write_text(yaml.safe_dump({**data_yaml, 'path': str(DRIVE_YOLO_ROOT)}, sort_keys=False), encoding='utf-8')

# Copy metadata and yaml to Drive; copy full dataset only if configured to avoid slow Drive writes.
COPY_SPLIT_DATASET_TO_DRIVE = False
if COPY_SPLIT_DATASET_TO_DRIVE:
    print('[copy full YOLO dataset to Drive] This can take time.')
    safe_copytree(LOCAL_YOLO_ROOT, DRIVE_YOLO_ROOT)
else:
    (DRIVE_YOLO_ROOT / 'README_local_dataset_note.txt').write_text(
        'Full split image/label folders were generated in Colab local runtime for speed. '\
        'Set COPY_SPLIT_DATASET_TO_DRIVE=True if you need persistent image/label copies. '\
        'Metadata CSV and data.yaml are persisted.\n', encoding='utf-8'
    )

required = []
for split in ['train', 'val', 'test']:
    img_count = len(list((LOCAL_YOLO_ROOT / 'images' / split).glob('*')))
    lbl_count = len(list((LOCAL_YOLO_ROOT / 'labels' / split).glob('*.txt')))
    required.append((split, img_count, lbl_count))
    print(split, 'images=', img_count, 'labels=', lbl_count)
    if img_count == 0 or lbl_count == 0:
        raise RuntimeError(f'{split} split is empty; check split generation.')

print('\nDATA_YAML_LOCAL')
print(DATA_YAML_LOCAL.read_text())
print('DATA_YAML_DRIVE:', DATA_YAML_DRIVE)

train images= 85039 labels= 85039
val images= 10636 labels= 10636
test images= 10757 labels= 10757

DATA_YAML_LOCAL
path: /content/anomali-road-safety-ai-work/datasets/plate_detection_yolo
train: images/train
val: images/val
test: images/test
nc: 1
names:
  0: license_plate

DATA_YAML_DRIVE: /content/drive/MyDrive/anomali-road-safety-ai/datasets/plate_detection_yolo/data.yaml


In [ ]:
# Cell 8 - Stage existing baseline checkpoint for comparison
BASELINE_LOCAL = LOCAL_ROOT / 'models' / 'checkpoints' / 'plate' / 'license_plate_detector.pt'
BASELINE_LOCAL.parent.mkdir(parents=True, exist_ok=True)

if BASELINE_PLATE_CHECKPOINT.exists():
    shutil.copy2(BASELINE_PLATE_CHECKPOINT, BASELINE_LOCAL)
    print('[baseline from Drive]', BASELINE_PLATE_CHECKPOINT, '->', BASELINE_LOCAL)
elif DOWNLOAD_BASELINE_IF_MISSING:
    print('[download baseline from HF]', BASELINE_HF_REPO, BASELINE_HF_FILENAME)
    from huggingface_hub import hf_hub_download
    downloaded = Path(hf_hub_download(repo_id=BASELINE_HF_REPO, filename=BASELINE_HF_FILENAME))
    shutil.copy2(downloaded, BASELINE_LOCAL)
    shutil.copy2(downloaded, BASELINE_PLATE_CHECKPOINT)
    print('baseline saved to Drive:', BASELINE_PLATE_CHECKPOINT)
else:
    print('[baseline missing] baseline comparison will be skipped')
    BASELINE_LOCAL = None

if BASELINE_LOCAL and BASELINE_LOCAL.exists():
    print('baseline size MB:', round(BASELINE_LOCAL.stat().st_size / 1024 / 1024, 2))

[download baseline from HF] morsetechlab/yolov11-license-plate-detection license-plate-finetune-v1n.pt



Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).


license-plate-finetune-v1n.pt:   0%|          | 0.00/5.47M [00:00<?, ?B/s]

baseline saved to Drive: /content/drive/MyDrive/anomali-road-safety-ai/models/checkpoints/plate/license_plate_detector.pt
baseline size MB: 5.21


In [ ]:
# Cell 9 - Train YOLO11n plate detector
from ultralytics import YOLO

train_start = time.time()
model = YOLO('yolo11n.pt')

train_results = model.train(
    data=str(DATA_YAML_LOCAL),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    workers=WORKERS,
    device=DEVICE,
    project=str(LOCAL_RUN_ROOT),
    name=EXPERIMENT_ID,
    exist_ok=True,
    seed=SEED,
    cache=CACHE_IMAGES,
    plots=True,
    verbose=True,
)
train_seconds = time.time() - train_start
print('train_seconds:', round(train_seconds, 2))

TRAIN_DIR = Path(getattr(train_results, 'save_dir', LOCAL_RUN_ROOT / EXPERIMENT_ID))
print('TRAIN_DIR:', TRAIN_DIR)
print('weights:', list((TRAIN_DIR / 'weights').glob('*.pt')))

Ultralytics 8.4.68 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=48, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/anomali-road-safety-ai-work/datasets/plate_detection_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=POCR-EXP-005-YOLO11N-PLATE-DETECTOR, nbs=64, nms=False, op

In [ ]:
# Cell 10 - Validation/test metrics for trained model and existing baseline
from ultralytics import YOLO

def metric_box_to_dict(metrics):
    box = getattr(metrics, 'box', None)
    out = {}
    if box is not None:
        for key in ['map', 'map50', 'map75', 'mp', 'mr']:
            try:
                out[key] = float(getattr(box, key))
            except Exception:
                out[key] = None
    try:
        out['fitness'] = float(getattr(metrics, 'fitness', None))
    except Exception:
        out['fitness'] = None
    return out

best_pt = TRAIN_DIR / 'weights' / 'best.pt'
last_pt = TRAIN_DIR / 'weights' / 'last.pt'
if not best_pt.exists():
    raise FileNotFoundError('best.pt not found: ' + str(best_pt))

trained_model = YOLO(str(best_pt))
metrics = {'trained': {}, 'baseline': {}}

for split in ['val', 'test']:
    print('\n[trained val]', split)
    m = trained_model.val(data=str(DATA_YAML_LOCAL), split=split, imgsz=IMGSZ, batch=BATCH, device=DEVICE, plots=True)
    metrics['trained'][split] = metric_box_to_dict(m)

if BASELINE_LOCAL and Path(BASELINE_LOCAL).exists():
    baseline_model = YOLO(str(BASELINE_LOCAL))
    for split in ['val', 'test']:
        print('\n[baseline val]', split)
        try:
            m = baseline_model.val(data=str(DATA_YAML_LOCAL), split=split, imgsz=IMGSZ, batch=BATCH, device=DEVICE, plots=False)
            metrics['baseline'][split] = metric_box_to_dict(m)
        except Exception as exc:
            metrics['baseline'][split] = {'error': repr(exc)}
            print('[baseline failed]', split, exc)
else:
    metrics['baseline']['status'] = 'missing'

print(json.dumps(metrics, indent=2))


[trained val] val
Ultralytics 8.4.68 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.1 ms, read: 564.5±284.0 MB/s, size: 19.5 KB)
val: Scanning /content/anomali-road-safety-ai-work/datasets/plate_detection_yolo/labels/val.cache... 10636 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 10636/10636 3.0Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 222/222 4.4it/s 51.0s
                   all      10636      11086      0.995      0.989      0.995      0.857
Speed: 0.5ms preprocess, 1.6ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /content/runs/detect/val

[trained val] test
Ultralytics 8.4.68 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 485.5±176.5 MB/s, size: 47.5 KB)
val: Scanning /c

In [ ]:
# Cell 11 - Optional UFPR-ALPR external benchmark preparation/evaluation
# UFPR is academic/non-commercial and access-controlled. This cell only runs if the user has placed the dataset in Drive.
UFPR_DRIVE_DIR = DRIVE_RAW_ROOT / 'ufpr_alpr'
UFPR_MANUAL_ZIP = DRIVE_RAW_ROOT / 'manual_zips' / 'ufpr_alpr.zip'
UFPR_LOCAL_DIR = LOCAL_RAW_ROOT / 'ufpr_alpr'
UFPR_YOLO_DIR = LOCAL_YOLO_ROOT / 'external_ufpr_alpr'
UFPR_DATA_YAML = UFPR_YOLO_DIR / 'data.yaml'


def parse_plate_bbox_from_txt(txt_path: Path):
    text = txt_path.read_text(encoding='utf-8', errors='ignore')
    # Common pattern: lines containing plate/lp/license and four integer coordinates.
    for line in text.splitlines():
        low = line.lower()
        if any(token in low for token in ['plate', 'license', 'lp']):
            nums = [float(x) for x in re.findall(r'-?\d+(?:\.\d+)?', line)]
            if len(nums) >= 4:
                # Prefer last four numbers in annotation-style lines.
                x1, y1, x2, y2 = nums[-4:]
                if x2 > x1 and y2 > y1:
                    return x1, y1, x2, y2
    return None


def prepare_ufpr_if_available():
    if UFPR_LOCAL_DIR.exists():
        pass
    elif UFPR_DRIVE_DIR.exists() and any(UFPR_DRIVE_DIR.rglob('*.png')):
        safe_copytree(UFPR_DRIVE_DIR, UFPR_LOCAL_DIR)
    elif UFPR_MANUAL_ZIP.exists():
        unzip_to(UFPR_MANUAL_ZIP, UFPR_LOCAL_DIR)
    else:
        return {'status': 'missing', 'message': 'UFPR zip/folder not found in Drive; external benchmark skipped.'}

    images = [p for p in UFPR_LOCAL_DIR.rglob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}]
    rows = []
    if UFPR_YOLO_DIR.exists():
        shutil.rmtree(UFPR_YOLO_DIR)
    for split in ['test']:
        (UFPR_YOLO_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
        (UFPR_YOLO_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

    for img in tqdm(images, desc='ufpr parse'):
        txt_candidates = [img.with_suffix('.txt'), img.parent / f'{img.stem}.txt']
        txt = next((p for p in txt_candidates if p.exists()), None)
        if txt is None:
            continue
        bbox = parse_plate_bbox_from_txt(txt)
        if bbox is None:
            continue
        try:
            with Image.open(img) as im:
                w, h = im.size
        except Exception:
            continue
        x1, y1, x2, y2 = bbox
        x1, x2 = max(0, min(x1, w)), max(0, min(x2, w))
        y1, y2 = max(0, min(y1, h)), max(0, min(y2, h))
        if x2 <= x1 or y2 <= y1:
            continue
        xc = ((x1 + x2) / 2) / w
        yc = ((y1 + y2) / 2) / h
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h
        stem = 'ufpr_' + hashlib.sha1(str(img).encode()).hexdigest()[:16]
        out_img = UFPR_YOLO_DIR / 'images' / 'test' / f'{stem}{img.suffix.lower()}'
        out_lbl = UFPR_YOLO_DIR / 'labels' / 'test' / f'{stem}.txt'
        shutil.copy2(img, out_img)
        out_lbl.write_text(f'0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n', encoding='utf-8')
        rows.append({'image': str(out_img), 'label': str(out_lbl), 'source': str(img)})

    if not rows:
        sample_txt = next(UFPR_LOCAL_DIR.rglob('*.txt'), None)
        return {
            'status': 'parse_failed',
            'message': 'UFPR files found, but bbox parser could not extract plate boxes. Inspect sample annotation.',
            'sample_annotation': str(sample_txt) if sample_txt else None,
        }

    UFPR_DATA_YAML.write_text(yaml.safe_dump({
        'path': str(UFPR_YOLO_DIR),
        'train': 'images/test',
        'val': 'images/test',
        'test': 'images/test',
        'nc': 1,
        'names': {0: 'license_plate'},
    }, sort_keys=False), encoding='utf-8')
    return {'status': 'ready', 'rows': len(rows), 'data_yaml': str(UFPR_DATA_YAML)}

ufpr_status = prepare_ufpr_if_available()
print('UFPR status:', ufpr_status)
ufpr_metrics = None
if ufpr_status.get('status') == 'ready':
    print('[UFPR external eval] trained model')
    m = trained_model.val(data=str(UFPR_DATA_YAML), split='test', imgsz=IMGSZ, batch=BATCH, device=DEVICE, plots=False)
    ufpr_metrics = metric_box_to_dict(m)
    print(ufpr_metrics)

UFPR status: {'status': 'missing', 'message': 'UFPR zip/folder not found in Drive; external benchmark skipped.'}


In [ ]:
# Cell 12 - Persist best/last checkpoints, optional ONNX, run folder, summary JSON, report
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

if best_pt.exists():
    shutil.copy2(best_pt, BEST_CHECKPOINT_DRIVE)
if last_pt.exists():
    shutil.copy2(last_pt, LAST_CHECKPOINT_DRIVE)

# Copy compact run artifacts to Drive. Avoid copying very large cache dirs.
safe_copytree(TRAIN_DIR, DRIVE_RUN_ROOT / TRAIN_DIR.name)

onnx_path = None
if RUN_ONNX_EXPORT:
    try:
        export_model = YOLO(str(best_pt))
        exported = export_model.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
        onnx_path = Path(exported)
        shutil.copy2(onnx_path, DRIVE_CHECKPOINT_ROOT / f'{EXPERIMENT_ID}-best.onnx')
        print('ONNX exported:', onnx_path)
    except Exception as exc:
        print('[ONNX export failed]', exc)
        onnx_path = None

summary = {
    'experiment_id': EXPERIMENT_ID,
    'generated_at_utc': now_utc(),
    'data_yaml_local': str(DATA_YAML_LOCAL),
    'data_yaml_drive': str(DATA_YAML_DRIVE),
    'metadata_csv_drive': str(METADATA_CSV_DRIVE),
    'best_checkpoint_drive': str(BEST_CHECKPOINT_DRIVE),
    'last_checkpoint_drive': str(LAST_CHECKPOINT_DRIVE) if LAST_CHECKPOINT_DRIVE.exists() else None,
    'onnx_checkpoint_drive': str(DRIVE_CHECKPOINT_ROOT / f'{EXPERIMENT_ID}-best.onnx') if onnx_path else None,
    'baseline_checkpoint_drive': str(BASELINE_PLATE_CHECKPOINT) if BASELINE_PLATE_CHECKPOINT.exists() else None,
    'train_seconds': train_seconds,
    'training_config': {
        'imgsz': IMGSZ, 'epochs': EPOCHS, 'patience': PATIENCE, 'batch': BATCH,
        'workers': WORKERS, 'device': DEVICE, 'seed': SEED,
    },
    'source_summaries': source_summaries,
    'split_distribution': metadata_df.groupby(['split', 'source_id']).agg(images=('image_name','count'), boxes=('bbox_count','sum')).reset_index().to_dict('records'),
    'metrics': metrics,
    'ufpr_status': ufpr_status,
    'ufpr_metrics': ufpr_metrics,
    'notes': [
        'Plate detection only; OCR not included.',
        'Roboflow datasets are merged after label normalization and dedup checks.',
        'UFPR-ALPR is skipped unless user-provided licensed dataset exists in Drive.',
    ],
}
SUMMARY_JSON_DRIVE.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

report_lines = [
    '# POCR-EXP-005 YOLO11n Plate Detector Report',
    '',
    f'Tarih: {now_utc()}',
    '',
    '## Amaç',
    '',
    'Turkish Number Plates + Roboflow LPR veri kaynaklarıyla tek sınıf `license_plate` YOLO11n plate detector fine-tune etmek ve mevcut lokal plate baseline ile karşılaştırmak.',
    '',
    '## Çıktılar',
    '',
    f'* Best checkpoint: `{BEST_CHECKPOINT_DRIVE}`',
    f'* Last checkpoint: `{LAST_CHECKPOINT_DRIVE if LAST_CHECKPOINT_DRIVE.exists() else "not_saved"}`',
    f'* ONNX: `{DRIVE_CHECKPOINT_ROOT / f"{EXPERIMENT_ID}-best.onnx" if onnx_path else "not_exported"}`',
    f'* Summary JSON: `{SUMMARY_JSON_DRIVE}`',
    f'* Metadata CSV: `{METADATA_CSV_DRIVE}`',
    '',
    '## Veri Kaynakları',
    '',
]
for src in source_summaries:
    report_lines.append(f"* `{src['source_id']}`: train={src.get('train',0)}, val={src.get('val',0)}, test={src.get('test',0)}")
report_lines += [
    '',
    '## Split Distribution',
    '',
    metadata_df.groupby(['split','source_id']).agg(images=('image_name','count'), boxes=('bbox_count','sum')).reset_index().to_markdown(index=False),
    '',
    '## Metrics',
    '',
    '```json',
    json.dumps(metrics, indent=2),
    '```',
    '',
    '## UFPR External Benchmark',
    '',
    '```json',
    json.dumps({'status': ufpr_status, 'metrics': ufpr_metrics}, indent=2),
    '```',
    '',
    '## Not',
    '',
    'Bu çalışma plate detection içindir. OCR doğruluğu veya hukuki/cezai karar iddiası üretmez.',
]
REPORT_MD_DRIVE.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')

print('SUMMARY:', SUMMARY_JSON_DRIVE)
print('REPORT:', REPORT_MD_DRIVE)
print('BEST:', BEST_CHECKPOINT_DRIVE)

Ultralytics 8.4.68 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/content/anomali-road-safety-ai-work/runs/plate_detection/POCR-EXP-005/POCR-EXP-005-YOLO11N-PLATE-DETECTOR/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 238ms
Prepared 4 packages in 1.57s
Installed 4 packages in 255ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.27.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 2.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to ta

## Çalıştırma Sonrası Kontrol Listesi

Notebook bittiğinde Drive'da şunları kontrol et:

1. `models/checkpoints/plate/POCR-EXP-005-YOLO11N-PLATE-DETECTOR-best.pt`
2. `models/checkpoints/plate/POCR-EXP-005-YOLO11N-PLATE-DETECTOR-best.onnx` (export başarılıysa)
3. `models/benchmarks/artifacts/plate_detection/POCR-EXP-005-YOLO11N-PLATE-DETECTOR-summary.json`
4. `models/experiments/POCR_EXP_005_plate_detector_report.md`
5. Val/test metriklerinde yeni modelin mevcut `license_plate_detector.pt` baseline'a göre üstün olup olmadığı.

UFPR için `ufpr_status.status == missing` normaldir. UFPR lisanslı zip'i Drive'a koyulduğunda aynı notebook tekrar çalıştırılabilir.